# 1) Overview - Exchange Grid Roundtrip (XML `<exchange>` rows)


This notebook is a wrapper around `nb04_xml_exchange_table_tool.py` to export/import ecospold `<exchange>` rows as a spreadsheet table.

## Which file do I edit?

- `outputs/04_exchanges_table.xlsx`: **authoritative import file** used by this notebook to write XML.
- `outputs/04_exchanges_table_with_process_matches.xlsx`: **decision-support file only** (suggested process names from matcher).

Important: this notebook imports only `TABLE_PATH` (default: `outputs/04_exchanges_table.xlsx`).
Changes made only in `outputs/04_exchanges_table_with_process_matches.xlsx` will **not** modify the XML.

## User Procedure (recommended)

1. Click **Regenerate Process Matches** (button cell) to rebuild `outputs/04_exchanges_table_with_process_matches.xlsx`, then review suggestions.
2. Decide which suggestions to accept.
3. Either apply choices manually in `outputs/04_exchanges_table.xlsx`, or set `APPLY_APPROVED_CHOICES=True` and run the auto-sync cell.
4. Confirm `__action=update` for changed rows (`add`/`delete` as needed for structural edits).
5. Run import cells in this notebook.
6. Validate the output XML.

## Important Rules

- Keep all helper columns (`__*`) in `outputs/04_exchanges_table.xlsx`.
- Do not rename or delete columns.
- Preserve one row per exchange unless you intentionally add/delete rows.
- Keep numeric fields numeric (`attr__meanValue`, etc.).
- If a value is unknown, leave the cell empty instead of writing free text like `na`.


In [11]:
# 2) Configure paths and roundtrip options
from pathlib import Path
import pandas as pd
from IPython.display import display

import sys
from pathlib import Path
sys.path.insert(0, str((((Path.cwd()) if (Path.cwd() / 'scripts').exists() else Path.cwd().parent) / 'scripts').resolve()))
import nb04_xml_exchange_table_tool as xgt

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'inputs').exists() and (PROJECT_DIR.parent / 'inputs').exists():
    PROJECT_DIR = PROJECT_DIR.parent
XML_PATH = PROJECT_DIR / 'inputs/04-05_Model_annual_crops.XML'
TABLE_PATH = PROJECT_DIR / 'outputs/04_exchanges_table.xlsx'  # authoritative import table (.csv or .xlsx)
MATCHES_PATH = PROJECT_DIR / 'outputs/04_exchanges_table_with_process_matches.xlsx'  # matcher review/approval table
CATALOG_PATH = PROJECT_DIR / 'inputs/04_lci_process_catalog.csv'
MATCH_TOP_N = 3
APPLY_APPROVED_CHOICES = True  # set True to copy approved_* choices into TABLE_PATH
IMPORTED_XML_PATH = PROJECT_DIR / 'inputs/04_Model_annual_crops_edited.XML'
RENUMBER_ON_IMPORT = True

assert XML_PATH.exists(), f'Missing XML: {XML_PATH}'
print('XML:', XML_PATH)
print('Table:', TABLE_PATH)
print('Matches:', MATCHES_PATH)
print('Catalog:', CATALOG_PATH)
print('Match top N:', MATCH_TOP_N)
print('Apply approved choices:', APPLY_APPROVED_CHOICES)
print('Import target:', IMPORTED_XML_PATH)


XML: Model_annual_crops.XML
Table: exchanges_table.xlsx
Matches: exchanges_table_with_process_matches.xlsx
Catalog: lci_process_catalog.csv
Match top N: 3
Apply approved choices: True
Import target: Model_annual_crops_edited.XML


In [ ]:
# 3) Regenerate process-match workbook from existing inputs/04_lci_process_catalog.csv
# Uses TABLE_PATH + CATALOG_PATH as inputs and writes MATCHES_PATH.

import ast
import importlib
import nb04_process_name_matcher as _pnm

_pnm = importlib.reload(_pnm)

RUN_MATCH_REFRESH_NOW = False  # fallback for frontends where widget buttons are not visible


def _emit_progress(cb, stage, current, total):
    if cb is None:
        return
    try:
        cb(stage, int(current), int(total))
    except Exception:
        pass


def _load_catalog_for_matching(catalog_path):
    catalog = pd.read_csv(catalog_path)
    for col in ['process_name','reference_product','location','unit','database','database_variant','category','extra_type','catalog_id','process_name_norm','reference_product_norm','unit_norm','location_norm']:
        if col not in catalog.columns:
            catalog[col] = ''

    if 'tokens' not in catalog.columns:
        catalog['tokens'] = [[] for _ in range(len(catalog))]
    else:
        def _parse_tokens(v):
            if isinstance(v, list):
                return v
            s = '' if pd.isna(v) else str(v).strip()
            if not s:
                return []
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, list):
                    return [str(x) for x in obj]
            except Exception:
                pass
            return [t.strip() for t in s.strip('[]').replace("'", '').split(',') if t.strip()]
        catalog['tokens'] = catalog['tokens'].map(_parse_tokens)

    if 'placeholder_comment' not in catalog.columns:
        catalog['placeholder_comment'] = ''

    return catalog


def _run_match_refresh(progress_cb=None):
    if not TABLE_PATH.exists():
        raise FileNotFoundError(f'Table not found: {TABLE_PATH}')
    if not CATALOG_PATH.exists():
        raise FileNotFoundError(f'Catalog not found: {CATALOG_PATH}')

    _emit_progress(progress_cb, 'load_catalog', 0, 1)
    catalog_df = _load_catalog_for_matching(CATALOG_PATH)
    _emit_progress(progress_cb, 'load_catalog', 1, 1)

    _emit_progress(progress_cb, 'load_exchanges', 0, 1)
    ex = pd.read_excel(TABLE_PATH) if TABLE_PATH.suffix.lower() in {'.xlsx', '.xls', '.xlsm'} else pd.read_csv(TABLE_PATH)
    ex = ex.rename(columns={c: str(c).strip() for c in ex.columns})
    if 'attr__location' in ex.columns:
        loc = ex['attr__location'].astype(str).replace({'nan': ''}).str.strip()
        ex = ex[loc.ne('')].copy()
    if 'attr__category' in ex.columns:
        cat = ex['attr__category'].astype(str).replace({'nan': ''}).str.strip().str.lower()
        ex = ex[~cat.isin({'air', 'water', 'soil'})].copy()
    _emit_progress(progress_cb, 'load_exchanges', 1, 1)

    review_rows = []
    total_rows = max(int(len(ex)), 1)
    _emit_progress(progress_cb, 'matching', 0, total_rows)

    for idx, (_, row) in enumerate(ex.iterrows(), start=1):
        name = str(row.get('attr__name', '') or '').strip()
        candidates = _pnm.suggest_matches_for_exchange(row, catalog_df, top_n=MATCH_TOP_N)

        base = row.to_dict()
        base['match_status'] = ''
        base['approved_database'] = ''
        base['approved_process_name'] = ''
        base['approved_location'] = ''
        base['approved_unit'] = ''
        base['match_notes'] = ''
        base['exchange_domain_hint'] = _pnm._exchange_domain_hint(row)
        base['match_candidate_count'] = len(candidates)

        for i in range(MATCH_TOP_N):
            n = i + 1
            if i < len(candidates):
                c = candidates[i]
                base[f'suggest_{n}_score'] = round(c.score, 4)
                base[f'suggest_{n}_database'] = c.database
                base[f'suggest_{n}_variant'] = c.variant
                base[f'suggest_{n}_process_name'] = c.process_name
                base[f'suggest_{n}_reference_product'] = c.reference_product
                base[f'suggest_{n}_location'] = c.location
                base[f'suggest_{n}_unit'] = c.unit
                base[f'suggest_{n}_reasons'] = c.reasons
            else:
                for suffix in ['score','database','variant','process_name','reference_product','location','unit','reasons']:
                    base[f'suggest_{n}_{suffix}'] = ''

        if candidates and candidates[0].score >= 0.92:
            base['match_status'] = 'auto_high_conf'
            base['approved_database'] = candidates[0].database
            base['approved_process_name'] = candidates[0].process_name
            base['approved_location'] = candidates[0].location
            base['approved_unit'] = candidates[0].unit
        elif not name:
            base['match_status'] = 'no_name'
        elif _pnm._exchange_domain_hint(row) == 'elementary':
            base['match_status'] = 'skip_elementary_flow'
        else:
            base['match_status'] = 'review'

        review_rows.append(base)
        _emit_progress(progress_cb, 'matching', idx, total_rows)

    review_df = pd.DataFrame(review_rows)

    summary = (
        review_df.groupby(['match_status', 'exchange_domain_hint'], dropna=False)
        .size()
        .reset_index(name='rows')
        .sort_values(['match_status', 'rows'], ascending=[True, False])
    )

    preview_cols = ['catalog_id','database','database_variant','process_name','reference_product','location','unit','category','extra_type']
    preview_cols = [c for c in preview_cols if c in catalog_df.columns]
    catalog_preview = catalog_df[preview_cols].copy()

    _emit_progress(progress_cb, 'write_workbook', 0, 1)
    with pd.ExcelWriter(MATCHES_PATH, engine='openpyxl') as writer:
        review_df.to_excel(writer, index=False, sheet_name='exchange_matches')
        summary.to_excel(writer, index=False, sheet_name='summary')
        catalog_preview.head(5000).to_excel(writer, index=False, sheet_name='catalog_preview')
    _emit_progress(progress_cb, 'write_workbook', 1, 1)
    _emit_progress(progress_cb, 'done', 1, 1)

    print(f'Regenerated matches: {MATCHES_PATH}')
    print(f'Catalog rows (from existing CSV): {len(catalog_df):,} | Match rows: {len(review_df):,}')
    if 'match_status' in review_df.columns:
        print(review_df['match_status'].value_counts(dropna=False).to_string())
    display(review_df.head(10))


try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    _btn = widgets.Button(
        description='Regenerate Process Matches',
        button_style='primary',
        icon='refresh',
        tooltip='Rebuild outputs/04_exchanges_table_with_process_matches.xlsx from current outputs/04_exchanges_table.xlsx and inputs/04_lci_process_catalog.csv',
    )
    _progress = widgets.IntProgress(value=0, min=0, max=1, description='Idle', bar_style='')
    _status = widgets.HTML(value='Ready to run.')
    _out = widgets.Output()

    _stage_labels = {
        'load_catalog': 'Catalog',
        'load_exchanges': 'Load',
        'matching': 'Matching',
        'write_workbook': 'Writing',
        'done': 'Done',
    }

    def _ui_progress(stage, current, total):
        total = max(int(total), 1)
        current = max(0, min(int(current), total))
        _progress.max = total
        _progress.value = current
        _progress.description = _stage_labels.get(stage, stage)
        _status.value = f'{_stage_labels.get(stage, stage)}: {current}/{total}'

    def _on_regenerate_click(_):
        _btn.disabled = True
        _btn.description = 'Running...'
        _btn.icon = 'hourglass'
        _progress.bar_style = 'info'
        _progress.value = 0
        _progress.max = 1
        _progress.description = 'Starting'
        _status.value = 'Starting regeneration...'

        with _out:
            clear_output(wait=True)
            try:
                _run_match_refresh(progress_cb=_ui_progress)
                _progress.bar_style = 'success'
                _progress.description = 'Done'
                _progress.value = _progress.max
                _status.value = 'Finished successfully.'
            except Exception as e:
                _progress.bar_style = 'danger'
                _progress.description = 'Error'
                _status.value = f'Failed: {e}'
                print(f'Error: {e}')
            finally:
                _btn.disabled = False
                _btn.description = 'Regenerate Process Matches'
                _btn.icon = 'refresh'

    _btn.on_click(_on_regenerate_click)
    display(_btn, _progress, _status, _out)
    print('If button is not visible in your notebook frontend, set RUN_MATCH_REFRESH_NOW=True and run this cell again.')

except Exception:
    print('ipywidgets UI not available. Set RUN_MATCH_REFRESH_NOW=True and run this cell to regenerate matches.')

if RUN_MATCH_REFRESH_NOW:
    _run_match_refresh()



Button(button_style='primary', description='Regenerate Process Matches', icon='refresh', style=ButtonStyle(), …

IntProgress(value=0, description='Idle', max=1)

HTML(value='Ready to run.')

Output()

If button is not visible in your notebook frontend, set RUN_MATCH_REFRESH_NOW=True and run this cell again.


In [16]:
# 4) Create/refresh outputs/04_exchanges_table.xlsx from XML (editable export)
# This step writes TABLE_PATH (default: outputs/04_exchanges_table.xlsx) from XML_PATH.
export_df = xgt.export_exchanges(XML_PATH, TABLE_PATH)
print(f'Exported {len(export_df):,} exchange rows to {TABLE_PATH}')
display(export_df.head(10))
print('Columns:', list(export_df.columns))


Exported 152 exchange rows to exchanges_table.xlsx


,__row_id,__action,__parent_path,__order,__tag,__ns_uri,__text,__children_xml,attr__category,attr__generalComment,attr__infrastructureProcess,attr__location,attr__meanValue,attr__name,attr__number,attr__subCategory,attr__uncertaintyType,attr__unit
0,ex_00001,keep,/ecoSpold[1]/dataset[1]/flowData[1],1,exchange,http://www.EcoInvent.org/EcoSpold01,,"<outputGroup xmlns=""http://www.EcoInvent.org/E...",NaN,"Model_annual_crops, yield per 1 ha, actual are...",false,GLO,666,simapro_process_name,1,NaN,NaN,kg
1,ex_00002,keep,/ecoSpold[1]/dataset[1]/flowData[1],2,exchange,http://www.EcoInvent.org/EcoSpold01,,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,NaN,false,GLO,0,"packaging, for fertilisers {GLO}| market for p...",2,NaN,0,kg
2,ex_00003,keep,/ecoSpold[1]/dataset[1]/flowData[1],3,exchange,http://www.EcoInvent.org/EcoSpold01,,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,NaN,false,GLO,0,"packaging, for pesticides {GLO}| market for pa...",3,NaN,0,kg
3,ex_00004,keep,/ecoSpold[1]/dataset[1]/flowData[1],4,exchange,http://www.EcoInvent.org/EcoSpold01,,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,NaN,false,GLO,0,"packaging, for fertilisers or pesticides {GLO}...",4,NaN,0,kg
4,ex_00005,keep,/ecoSpold[1]/dataset[1]/flowData[1],5,exchange,http://www.EcoInvent.org/EcoSpold01,,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,LDPE,false,GLO,0,"packaging film, LDPE, at plant {RER} U",5,NaN,0,kg
5,ex_00006,keep,/ecoSpold[1]/dataset[1]/flowData[1],6,exchange,http://www.EcoInvent.org/EcoSpold01,,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,PE,false,GLO,0,"packaging film, PE, at plant {RER} U",6,NaN,0,kg
6,ex_00007,keep,/ecoSpold[1]/dataset[1]/flowData[1],7,exchange,http://www.EcoInvent.org/EcoSpold01,,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,PP,false,GLO,0,"packaging film, PP, at plant {RER} U",7,NaN,0,kg
7,ex_00008,keep,/ecoSpold[1]/dataset[1]/flowData[1],8,exchange,http://www.EcoInvent.org/EcoSpold01,,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,PET,false,GLO,0,"packaging film, PET, at plant {RER} U",8,NaN,0,kg
8,ex_00009,keep,/ecoSpold[1]/dataset[1]/flowData[1],9,exchange,http://www.EcoInvent.org/EcoSpold01,,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,Kraft,false,GLO,0,kraft paper {RER}| market for kraft paper | Cu...,9,NaN,0,kg
9,ex_00010,keep,/ecoSpold[1]/dataset[1]/flowData[1],10,exchange,http://www.EcoInvent.org/EcoSpold01,,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,Cardboard,false,GLO,0,"cardboard box, at plant {RER} U",10,NaN,0,kg


Columns: ['__row_id', '__action', '__parent_path', '__order', '__tag', '__ns_uri', '__text', '__children_xml', 'attr__category', 'attr__generalComment', 'attr__infrastructureProcess', 'attr__location', 'attr__meanValue', 'attr__name', 'attr__number', 'attr__subCategory', 'attr__uncertaintyType', 'attr__unit']


## 5) Editing Guide - what to change and how

In this step, **manually edit** `outputs/04_exchanges_table_with_process_matches.xlsx`.

What to do here:
- Review `suggest_1_*`, `suggest_2_*`, `suggest_3_*` columns.
- Enter your accepted choice in `approved_process_name` (and optionally `approved_unit`, `approved_location`).
- Set `__action` for each row according to your intent.
`keep`: keep the exchange unchanged.
`update`: modify an existing exchange (most common when accepting a suggested process).
`add`: add a new exchange row.
`delete`: remove an existing exchange row.
- Keep helper identifiers like `__row_id` unchanged.
- Save the file after your edits.

Important:
- Do **not** expect `outputs/04_exchanges_table.xlsx` to change in this step.
- The transfer of your approved edits from `outputs/04_exchanges_table_with_process_matches.xlsx` into `outputs/04_exchanges_table.xlsx` happens in **Step 6**.



In [17]:
# 6) Apply approved process matches into the main exchange table
# Source workbook: outputs/04_exchanges_table_with_process_matches.xlsx
# Target workbook: outputs/04_exchanges_table.xlsx
if APPLY_APPROVED_CHOICES:
    if not MATCHES_PATH.exists():
        raise FileNotFoundError(f'Matches file not found: {MATCHES_PATH}')
    if not TABLE_PATH.exists():
        raise FileNotFoundError(f'Table file not found: {TABLE_PATH}')

    matches_df = pd.read_excel(MATCHES_PATH) if MATCHES_PATH.suffix.lower() in {'.xlsx', '.xls', '.xlsm'} else pd.read_csv(MATCHES_PATH)
    table_df = pd.read_excel(TABLE_PATH) if TABLE_PATH.suffix.lower() in {'.xlsx', '.xls', '.xlsm'} else pd.read_csv(TABLE_PATH)

    required = {'__row_id', 'approved_process_name'}
    missing = required - set(matches_df.columns)
    if missing:
        raise ValueError(f'Matches file missing required columns: {sorted(missing)}')
    if '__row_id' not in table_df.columns:
        raise ValueError('Target table is missing __row_id')

    approved = matches_df.copy()
    approved['approved_process_name'] = approved['approved_process_name'].fillna('').astype(str).str.strip()
    approved = approved[approved['approved_process_name'].ne('')].copy()

    if approved.empty:
        print('No approved_process_name values found. No updates applied.')
    else:
        # keep only the last approval per row_id if duplicates exist
        approved = approved.drop_duplicates(subset=['__row_id'], keep='last')
        approved = approved.set_index('__row_id')

        updated_rows = 0
        name_updates = 0
        unit_updates = 0
        loc_updates = 0

        table_df = table_df.copy()
        if '__action' not in table_df.columns:
            table_df['__action'] = 'keep'

        for i, row in table_df.iterrows():
            rid = row.get('__row_id')
            if rid not in approved.index:
                continue

            ap = approved.loc[rid]
            changed = False

            new_name = str(ap.get('approved_process_name', '') or '').strip()
            old_name = str(row.get('attr__name', '') or '').strip()
            if new_name and new_name != old_name:
                table_df.at[i, 'attr__name'] = new_name
                name_updates += 1
                changed = True

            if 'approved_unit' in approved.columns and 'attr__unit' in table_df.columns:
                new_unit = str(ap.get('approved_unit', '') or '').strip()
                old_unit = str(row.get('attr__unit', '') or '').strip()
                if new_unit and new_unit != old_unit:
                    table_df.at[i, 'attr__unit'] = new_unit
                    unit_updates += 1
                    changed = True

            if 'approved_location' in approved.columns and 'attr__location' in table_df.columns:
                new_loc = str(ap.get('approved_location', '') or '').strip()
                old_loc = str(row.get('attr__location', '') or '').strip()
                if new_loc and new_loc != old_loc:
                    table_df.at[i, 'attr__location'] = new_loc
                    loc_updates += 1
                    changed = True

            if changed:
                updated_rows += 1
                action = str(row.get('__action', '') or '').strip().lower()
                if action in {'', 'keep', 'nan'}:
                    table_df.at[i, '__action'] = 'update'

        if TABLE_PATH.suffix.lower() in {'.xlsx', '.xls', '.xlsm'}:
            table_df.to_excel(TABLE_PATH, index=False)
        else:
            table_df.to_csv(TABLE_PATH, index=False)

        print('Applied approved choices to target table:')
        print({'rows_touched': updated_rows, 'name_updates': name_updates, 'unit_updates': unit_updates, 'location_updates': loc_updates})
        display(table_df[table_df['__row_id'].isin(approved.index)].head(10))
else:
    print('APPLY_APPROVED_CHOICES=False -> skipping auto-sync from matches file.')


Applied approved choices to target table:
{'rows_touched': 7, 'name_updates': 7, 'unit_updates': 6, 'location_updates': 7}


,__row_id,__action,__parent_path,__order,__tag,__ns_uri,__text,__children_xml,attr__category,attr__generalComment,attr__infrastructureProcess,attr__location,attr__meanValue,attr__name,attr__number,attr__subCategory,attr__uncertaintyType,attr__unit
18,ex_00019,update,/ecoSpold[1]/dataset[1]/flowData[1],19,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,"Cattle, solid",False,nan,0,"Manure, from cattle, stocked in concrete surfa...",19,NaN,0.0,nan
19,ex_00020,update,/ecoSpold[1]/dataset[1]/flowData[1],20,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,"Cattle, liquid",False,nan,0,"Slurry, from cattle, stocked in silo (fertiliz...",20,NaN,0.0,nan
20,ex_00021,update,/ecoSpold[1]/dataset[1]/flowData[1],21,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,"Swine, liquid",False,nan,0,"Slurry, from swine, stocked in silo (fertilize...",21,NaN,0.0,nan
21,ex_00022,update,/ecoSpold[1]/dataset[1]/flowData[1],22,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,"Poultry, droppings",False,nan,0,"Manure, from poultry, stocked in concrete surf...",22,NaN,0.0,nan
22,ex_00023,update,/ecoSpold[1]/dataset[1]/flowData[1],23,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,Compost,False,nan,0,"Average compost, from green waste, biowaste, s...",23,NaN,0.0,nan
23,ex_00024,update,/ecoSpold[1]/dataset[1]/flowData[1],24,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,Compost_digestate,False,RER,0,"Compost, of solid fraction of digestate from m...",24,NaN,0.0,kg
24,ex_00025,update,/ecoSpold[1]/dataset[1]/flowData[1],25,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,Digestate,False,nan,0,Average agricultural digestate (fertilizer) {R...,25,NaN,0.0,nan


In [18]:
# 7) Preview edited table before XML import
if TABLE_PATH.exists():
    edited_df = pd.read_excel(TABLE_PATH) if TABLE_PATH.suffix.lower() in {'.xlsx', '.xls', '.xlsm'} else pd.read_csv(TABLE_PATH)
    print(f'Edited table shape: {edited_df.shape}')
    display(edited_df.head(10))
    if '__action' in edited_df.columns:
        display(edited_df['__action'].fillna('keep').astype(str).value_counts(dropna=False).rename('rows'))
else:
    print('Table file not found yet:', TABLE_PATH)


Edited table shape: (152, 18)


,__row_id,__action,__parent_path,__order,__tag,__ns_uri,__text,__children_xml,attr__category,attr__generalComment,attr__infrastructureProcess,attr__location,attr__meanValue,attr__name,attr__number,attr__subCategory,attr__uncertaintyType,attr__unit
0,ex_00001,keep,/ecoSpold[1]/dataset[1]/flowData[1],1,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<outputGroup xmlns=""http://www.EcoInvent.org/E...",NaN,"Model_annual_crops, yield per 1 ha, actual are...",0.0,GLO,666,simapro_process_name,1,NaN,NaN,kg
1,ex_00002,keep,/ecoSpold[1]/dataset[1]/flowData[1],2,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,NaN,0.0,GLO,0,"packaging, for fertilisers {GLO}| market for p...",2,NaN,0.0,kg
2,ex_00003,keep,/ecoSpold[1]/dataset[1]/flowData[1],3,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,NaN,0.0,GLO,0,"packaging, for pesticides {GLO}| market for pa...",3,NaN,0.0,kg
3,ex_00004,keep,/ecoSpold[1]/dataset[1]/flowData[1],4,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,NaN,0.0,GLO,0,"packaging, for fertilisers or pesticides {GLO}...",4,NaN,0.0,kg
4,ex_00005,keep,/ecoSpold[1]/dataset[1]/flowData[1],5,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,LDPE,0.0,GLO,0,"packaging film, LDPE, at plant {RER} U",5,NaN,0.0,kg
5,ex_00006,keep,/ecoSpold[1]/dataset[1]/flowData[1],6,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,PE,0.0,GLO,0,"packaging film, PE, at plant {RER} U",6,NaN,0.0,kg
6,ex_00007,keep,/ecoSpold[1]/dataset[1]/flowData[1],7,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,PP,0.0,GLO,0,"packaging film, PP, at plant {RER} U",7,NaN,0.0,kg
7,ex_00008,keep,/ecoSpold[1]/dataset[1]/flowData[1],8,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,PET,0.0,GLO,0,"packaging film, PET, at plant {RER} U",8,NaN,0.0,kg
8,ex_00009,keep,/ecoSpold[1]/dataset[1]/flowData[1],9,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,Kraft,0.0,GLO,0,kraft paper {RER}| market for kraft paper | Cu...,9,NaN,0.0,kg
9,ex_00010,keep,/ecoSpold[1]/dataset[1]/flowData[1],10,exchange,http://www.EcoInvent.org/EcoSpold01,NaN,"<inputGroup xmlns=""http://www.EcoInvent.org/Ec...",NaN,Cardboard,0.0,GLO,0,"cardboard box, at plant {RER} U",10,NaN,0.0,kg


__action
keep      145
update      7
Name: rows, dtype: int64

In [19]:
# 8) Import edited table back into a new XML file
updated, removed, added = xgt.import_exchanges(
    xml_in=XML_PATH,
    table_in=TABLE_PATH,
    xml_out=IMPORTED_XML_PATH,
    renumber=RENUMBER_ON_IMPORT,
)
print('Wrote:', IMPORTED_XML_PATH)
print({'updated': updated, 'removed': removed, 'added': added, 'renumber': RENUMBER_ON_IMPORT})


Wrote: Model_annual_crops_edited.XML
{'updated': 152, 'removed': 0, 'added': 0, 'renumber': True}


In [20]:
# 9) Run quick sanity checks on the imported XML
if IMPORTED_XML_PATH.exists():
    with IMPORTED_XML_PATH.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i > 120:
                break
            print(line.rstrip())


<?xml version='1.0' encoding='UTF-8'?>
<ecoSpold xmlns="http://www.EcoInvent.org/EcoSpold01" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.EcoInvent.org/EcoSpold01 http://www.EcoInvent.org/EcoSpold01..\..\EcoSpold01Dataset.xsd">
  <dataset validCompanyCodes="CompanyCodes.xml" validRegionalCodes="RegionalCodes.xml" validCategories="Categories.xml" validUnits="Units.xml" number="1" timestamp="2025-11-25T14:05:27" generator="SimaPro 9.5.0.2">
    <metaInformation>
      <processInformation>
        <referenceFunction datasetRelatesToProduct="true" name="simapro_process_name" localName="simapro_process_name" infrastructureProcess="false" amount="666" unit="kg" category="Agricultural" subCategory="Mas Badia" localCategory="Agricultural" localSubCategory="Mas Badia" infrastructureIncluded="false" />
        <geography location="GLO" text="Unspecified" />
        <technology text="Unspecified" />
        <timePeriod dataValidForEntirePeriod="" text="Unsp

## 10) Quick Start - non-technical checklist


1. Review suggestions in `outputs/04_exchanges_table_with_process_matches.xlsx`.
2. Copy accepted choices into `outputs/04_exchanges_table.xlsx` (same rows).
3. In `outputs/04_exchanges_table.xlsx`, set `__action` to `update`/`add`/`delete` as needed.
4. Run import cells (they read `outputs/04_exchanges_table.xlsx`, not the matches file).
5. Open the new XML and verify edited exchanges are correct.
